**Document Ingestion**

**Installing the dependencies**

In [1]:
!pip install -U -q -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Importing the dependencies**

In [ ]:
from dotenv import load_dotenv

# 1. Load your API key from .env
load_dotenv()


#so we use this settings to set up my llm using .setting file
from llama_index.core import Settings

#simple directory -> we just need to give the path of directory and it will automatically read the diffrent files
#Storage Context -> wrapper around the vector store
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext

#chroma vector db
from llama_index.vector_stores.chroma import ChromaVectorStore

#SimpleNodeParser -> to chunk the text and add over lap and all other paramaters 
from llama_index.core.node_parser import SimpleNodeParser

# WE will use open ai embedding model  
from llama_index.embeddings.openai import OpenAIEmbedding
#from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import chromadb

This nltk modules gonna internally used 

In [ ]:
import nltk
import ssl

nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
# configuration
docs_dir_path = r"C:\Users\matule\OneDrive - Capgemini\Desktop\Training\Agentic Ai\RAG_Llama_Index\docs_dir"

#this vector folder will be created automatically
vector_db_path = r"C:\Users\matule\OneDrive - Capgemini\Desktop\Training\Agentic Ai\RAG_Llama_Index\vector_db"
collection_name="documents_collection"

**Document Ingestion**

In [ ]:
# define embedding function
#embed_model = HuggingFaceEmbedding()


# Define OpenAI embedding pointing to Capgemini Enterprise Gateway
embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_base="https://openai.generative.engine.capgemini.com/v1"
)

# Set as default embedding model
Settings.embed_model = embed_model


Setting up the loader

In [ ]:
# directory loader
loader = SimpleDirectoryReader(input_dir=docs_dir_path)

Loading the documents

In [ ]:
# load the documents
documents = loader.load_data()

In [10]:
print(len(documents))

28


In [ ]:
documents[0]

In [ ]:
documents[1]

In [ ]:
# Define persistent DB location

# we are connecting vector db using persistent client
db = chromadb.PersistentClient(path=vector_db_path)

In [ ]:
# If the collection is not present it will create it and if present it will load it 
chroma_collection = db.get_or_create_collection(name=collection_name)

We can play around the values of chunk size and overlap size so we can maintain continuity.

In [22]:
# Create parser with chunking strategy
parser = SimpleNodeParser.from_defaults(chunk_size=1024, chunk_overlap=50)

Creating the nodes(chunks) by passing the documents 

In [23]:
# Convert documents to chunks (nodes)
nodes = parser.get_nodes_from_documents(documents)

Creating the vector store using by giving the collection name

In [24]:
# Create vector store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

Storage context is just like a wrapper around the vector store so if we 
want to change our vector database we can directly do it by changing the name here in the parameter of this StorageContext function.


In [25]:
# Create storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In langchain we created retriver its similar to like that

Index is like table of contents for the storage context 

In LlamaIndex, a RAG system stores more than just vectors. It stores:

The Vectors (numbers for similarity search).
The Raw Text & Chunks (the actual words).
The Graph/Node Relationships (which chunk came from which page/document).
StorageContext is the container that holds all three of these stores together.

In [8]:
# Create the vector store index
index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context, 

    #vector store parameter is redundant here
    vector_store=vector_store, 
    embed_model=embed_model
)

NameError: name 'nodes' is not defined

In [ ]:
print("vector database created")